# Model 1: EfficientNetV2-S for Multi-Pathology Eye Disease Detection

This notebook covers training, fine-tuning, weight loading, and inference benchmarking for **EfficientNetV2-S** on fundus eye images across 6 classes: `AMD`, `Cataract`, `Dementia`, `Diabetes`, `Glaucoma`, `Normal`.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import timm

CLASSES = ['AMD', 'Cataract', 'Dementia', 'Diabetes', 'Glaucoma', 'Normal']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHT_PATH = 'weights/efficientnetv2s_scratch_best.pth'

print(f"Using PyTorch Version: {torch.__version__}")
print(f"Execution Device: {DEVICE}")

## 1. Load EfficientNetV2-S Architecture & Weights

In [ ]:
def build_efficientnetv2s(num_classes=6):
    model = timm.create_model('tf_efficientnetv2_s', pretrained=False, num_classes=num_classes)
    return model

model = build_efficientnetv2s(len(CLASSES))
if os.path.exists(WEIGHT_PATH):
    state_dict = torch.load(WEIGHT_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print(f"Successfully loaded weight file from: {WEIGHT_PATH}")
else:
    print(f"Weight file not found at {WEIGHT_PATH}, using uninitialized model.")

model = model.to(DEVICE)
model.eval()

## 2. Image Preprocessing & Sample Inference

In [ ]:
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dummy test tensor for verification
dummy_input = torch.randn(1, 3, 640, 640).to(DEVICE)
with torch.no_grad():
    output = model(dummy_input)
    probabilities = torch.softmax(output, dim=1)
    top_prob, top_class = torch.topk(probabilities, 1)

print(f"Predicted Class Index: {top_class.item()} ({CLASSES[top_class.item()]})")
print(f"Confidence: {top_prob.item() * 100:.2f}%")